# Notebook 4: Computational Exploration
## Hadwiger's Conjecture — Beautiful Dead Ends

---

### The computational question

Three notebooks of theory have established what the conjecture says, what minors are, and why k≥7 resists the proof strategies that worked for smaller cases. This notebook turns computational.

The questions I want to answer:
1. Does the conjecture hold on random graphs? How many trials before a violation appears?
2. What do the **tight cases** look like — graphs where h(G) = χ(G) exactly, neither comfortably above?
3. What are the **adversarial cases** — graphs designed to stress the conjecture?
4. Where exactly does the computational search become intractable?

The answers tell us something about the *nature* of the difficulty — whether it's the conjecture that's hard, or just the computation.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product as iproduct, combinations
import random
import time

plt.style.use('dark_background')
ACCENT  = '#e05c5c'
NEUTRAL = '#888888'
WHITE   = '#dddddd'
COLOUR_PALETTE = ['#e05c5c','#5c9ee0','#5ce08a','#e0a05c',
                  '#c05ce0','#e0e05c','#5ce0d8','#e05ca0']

# ── Core functions ────────────────────────────────────────────────────────
def complete_graph(n):
    g = np.ones((n,n), dtype=int); np.fill_diagonal(g, 0); return g

def cycle_graph(n):
    g = np.zeros((n,n), dtype=int)
    for i in range(n): g[i][(i+1)%n] = g[(i+1)%n][i] = 1
    return g

def random_graph(n, p, seed=None):
    if seed is not None: random.seed(seed)
    g = np.zeros((n,n), dtype=int)
    for i in range(n):
        for j in range(i+1,n):
            if random.random() < p: g[i][j] = g[j][i] = 1
    return g

def contract_edge(adj, u, v):
    n = len(adj)
    mapping = {i: sum(1 for j in range(i) if j != v) for i in range(n) if i != v}
    new_adj = np.zeros((n-1,n-1), dtype=int)
    for i in range(n):
        for j in range(n):
            if i != v and j != v:
                new_adj[mapping[i]][mapping[j]] = adj[i][j]
    mu = mapping[u]
    for w in range(n):
        if w != v and w != u and adj[v][w]:
            new_adj[mu][mapping[w]] = new_adj[mapping[w]][mu] = 1
    return new_adj

def delete_vertex(adj, v):
    idx = [i for i in range(len(adj)) if i != v]
    return adj[np.ix_(idx, idx)]

def has_k_minor(adj, k, depth=0, max_depth=12):
    n = len(adj)
    if n < k: return False
    if n == k:
        return np.array_equal(adj, np.ones((k,k),dtype=int)-np.eye(k,dtype=int))
    if depth >= max_depth: return False
    for u in range(n):
        for v in range(u+1,n):
            if adj[u][v] and has_k_minor(contract_edge(adj,u,v),k,depth+1,max_depth):
                return True
    for v in range(n):
        if has_k_minor(delete_vertex(adj,v),k,depth+1,max_depth): return True
    return False

def chromatic_number(adj, max_k=7):
    n = len(adj)
    for k in range(1, max_k+1):
        for col in iproduct(range(k), repeat=n):
            if all(not adj[u][v] or col[u]!=col[v]
                   for u in range(n) for v in range(u+1,n)):
                return k, list(col)
    return max_k, None

def hadwiger_number(adj, max_k=7):
    for k in range(max_k, 0, -1):
        if has_k_minor(adj, k): return k
    return 1

def mycielski(g):
    """Mycielski construction: increases chi by 1, preserves triangle-free."""
    n = len(g); new_n = 2*n + 1
    new_g = np.zeros((new_n, new_n), dtype=int)
    new_g[:n,:n] = g
    for i in range(n):
        for j in range(n):
            if g[i][j]: new_g[n+i][j] = new_g[j][n+i] = 1
    for i in range(n): new_g[2*n][n+i] = new_g[n+i][2*n] = 1
    return new_g

def draw_graph(adj, colouring=None, title='', ax=None, pos=None):
    n = len(adj)
    if ax is None: fig, ax = plt.subplots(figsize=(5,5))
    if pos is None:
        angles = np.linspace(0, 2*np.pi, n, endpoint=False)
        pos = np.column_stack([np.cos(angles), np.sin(angles)])
    for u in range(n):
        for v in range(u+1,n):
            if adj[u][v]:
                ax.plot([pos[u,0],pos[v,0]],[pos[u,1],pos[v,1]],
                        color=NEUTRAL, linewidth=1, alpha=0.5, zorder=1)
    for v in range(n):
        c = COLOUR_PALETTE[colouring[v]] if colouring else WHITE
        ax.scatter(pos[v,0],pos[v,1],s=280,color=c,zorder=3,
                   edgecolors='white',linewidth=1)
        ax.text(pos[v,0],pos[v,1],str(v),ha='center',va='center',
                fontsize=7,color='black',fontweight='bold',zorder=4)
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5)
    ax.set_aspect('equal'); ax.axis('off')
    if title: ax.set_title(title, fontsize=9, pad=6)

print('Setup complete.')

## 1. Random graph trials — searching for a counterexample

If Hadwiger's conjecture is false, there exists a graph G where χ(G) > h(G) — a graph that needs more colours than its largest complete minor would suggest. No such graph has ever been found. Let me try.

In [ ]:
# Search for violations across random graphs
# Also track: tight cases (h = chi), comfortable cases (h > chi)

N_TRIALS = 100
N_VERTICES = 8
P_EDGE = 0.5

violations = []
tight = []       # h(G) == chi(G)
comfortable = [] # h(G) > chi(G)

print(f'Testing {N_TRIALS} random graphs (n={N_VERTICES}, p={P_EDGE})...')
t0 = time.time()

for seed in range(N_TRIALS):
    g = random_graph(N_VERTICES, P_EDGE, seed)
    chi, col = chromatic_number(g, max_k=6)
    h = hadwiger_number(g, max_k=6)
    gap = h - chi
    if gap < 0:
        violations.append((seed, chi, h, g))
    elif gap == 0:
        tight.append((seed, chi, h, g))
    else:
        comfortable.append((seed, chi, h, g))

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')
print()
print(f'Violations (h < chi):   {len(violations)}')
print(f'Tight (h = chi):        {len(tight)}')
print(f'Comfortable (h > chi):  {len(comfortable)}')
print()

from collections import Counter
print('Distribution of chi values across all trials:')
all_results = [(s,c,h,g) for s,c,h,g in tight+comfortable]
chi_dist = Counter(c for _,c,_,_ in all_results)
for chi_val in sorted(chi_dist):
    print(f'  chi={chi_val}: {chi_dist[chi_val]} graphs')

In [ ]:
# Visualise the gap distribution
gaps = [h - chi for _, chi, h, _ in tight + comfortable]
chi_vals = [chi for _, chi, _, _ in tight + comfortable]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Hadwiger gap h(G) − χ(G) across {N_TRIALS} random graphs (n={N_VERTICES}, p={P_EDGE})',
             fontsize=11)

ax = axes[0]
from collections import Counter
gap_counts = Counter(gaps)
gap_vals = sorted(gap_counts.keys())
colours = [ACCENT if g == 0 else NEUTRAL for g in gap_vals]
ax.bar(gap_vals, [gap_counts[g] for g in gap_vals], color=colours, alpha=0.8, edgecolor='none')
ax.set_xlabel('h(G) − χ(G)', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('Gap distribution\nRed = tight cases (gap=0)', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
ax.scatter(chi_vals, gaps, color=ACCENT, alpha=0.5, s=30, edgecolors='none')
ax.axhline(0, color=WHITE, linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('χ(G)', fontsize=10)
ax.set_ylabel('h(G) − χ(G)', fontsize=10)
ax.set_title('Gap vs chromatic number\nNo violations — always ≥ 0', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print('The conjecture holds in every case. The gap is never negative.')

## 2. The tight cases — where the conjecture is most constrained

Tight cases — where h(G) = χ(G) exactly — are the graphs that give the conjecture the least room to breathe. In a comfortable case, h(G) is much larger than χ(G), so the conjecture is satisfied with room to spare. Tight cases are where one fewer contraction or one more colour would make things interesting.

What do tight cases look like structurally?

In [ ]:
# Filter tight cases with chi >= 4 — the non-trivial ones
tight_interesting = [(s,c,h,g) for s,c,h,g in tight if c >= 4]
print(f'Tight cases with chi >= 4: {len(tight_interesting)}')
print()

# Properties of tight vs comfortable cases
def graph_stats(g):
    n = len(g)
    edges = g.sum() // 2
    avg_deg = g.sum() / n
    min_deg = g.sum(axis=1).min()
    return {'n': n, 'edges': edges, 'avg_deg': avg_deg, 'min_deg': min_deg}

tight_chi4 = [(s,c,h,g) for s,c,h,g in tight if c == 4]
comfortable_chi4 = [(s,c,h,g) for s,c,h,g in comfortable if c == 4]

if tight_chi4 and comfortable_chi4:
    tight_avg_deg    = np.mean([graph_stats(g)['avg_deg'] for _,_,_,g in tight_chi4])
    comf_avg_deg     = np.mean([graph_stats(g)['avg_deg'] for _,_,_,g in comfortable_chi4])
    tight_avg_edges  = np.mean([graph_stats(g)['edges']   for _,_,_,g in tight_chi4])
    comf_avg_edges   = np.mean([graph_stats(g)['edges']   for _,_,_,g in comfortable_chi4])

    print('Comparison of tight vs comfortable cases (chi=4):')
    print(f'                     Tight (h=4)   Comfortable (h>4)')
    print(f'Count:               {len(tight_chi4):>11}   {len(comfortable_chi4):>16}')
    print(f'Avg edges:           {tight_avg_edges:>11.1f}   {comf_avg_edges:>16.1f}')
    print(f'Avg degree:          {tight_avg_deg:>11.2f}   {comf_avg_deg:>16.2f}')
    print()
    print('Tight cases tend to be sparser — the minimum minor structure')
    print('needed to satisfy the conjecture with nothing to spare.')

In [ ]:
# Draw some tight cases
if tight_interesting:
    n_show = min(4, len(tight_interesting))
    fig, axes = plt.subplots(1, n_show, figsize=(13, 4))
    fig.suptitle('Tight cases: h(G) = χ(G) — the conjecture satisfied with nothing to spare',
                 fontsize=11)
    if n_show == 1: axes = [axes]

    for ax, (seed, chi, h, g) in zip(axes, tight_interesting[:n_show]):
        _, col = chromatic_number(g, max_k=chi+1)
        draw_graph(g, col, f'seed={seed}\nχ={chi}, h={h}', ax=ax)

    plt.tight_layout()
    plt.show()

## 3. Mycielski graphs — the adversarial cases

Random graphs are useful but not adversarial. The most challenging graphs for Hadwiger's Conjecture are those that have **high chromatic number but low density** — graphs that need many colours despite being sparse.

The **Mycielski construction** produces exactly these. Starting from K₂, each application of the construction increases the chromatic number by 1 while preserving the triangle-free property. The result is a family of graphs with:
- χ grows without bound
- No triangle (so no K₃ subgraph — the minor has to come purely from contractions)
- Relatively sparse edge sets

These are the graphs most likely to stress Hadwiger's Conjecture. If a counterexample exists, it would probably look like a Mycielski-type graph.

In [ ]:
# Build Mycielski family
k2 = complete_graph(2)
m3 = mycielski(k2)   # C5: n=5,  chi=3
m4 = mycielski(m3)   # Grötzsch: n=11, chi=4
m5 = mycielski(m4)   # n=23, chi=5

def has_triangle(adj):
    n = len(adj)
    for u,v,w in combinations(range(n), 3):
        if adj[u][v] and adj[v][w] and adj[u][w]: return True
    return False

print('Mycielski graph family:')
print(f'{"Graph":12s}  {"n":>4}  {"edges":>7}  {"avg deg":>8}  {"Triangle-free":>14}  chi (expected)')
print('-' * 68)

for name, g, expected_chi in [('M3 (C5)',   m3, 3),
                                ('M4 (Grötzsch)', m4, 4),
                                ('M5',        m5, 5)]:
    n = len(g)
    edges = g.sum()//2
    avg_d = g.sum()/n
    tri_free = not has_triangle(g)
    print(f'{name:12s}  {n:>4}  {edges:>7}  {avg_d:>8.2f}  {str(tri_free):>14}  {expected_chi}')

print()
print('Each step: triangle-free preserved, chi increases by 1, size more than doubles.')
print('These graphs have high chi despite being sparse — maximum stress on the conjecture.')

In [ ]:
# Test Hadwiger on M3 and M4 — verify conjecture holds
# M5 is too large for chromatic_number but we can check Hadwiger number

print('Hadwiger conjecture on Mycielski graphs:')
print(f'{"Graph":12s}  {"n":>4}  {"χ":>4}  {"h(G)":>6}  {"h≥χ?":>8}  time')
print('-' * 52)

for name, g, expected_chi in [('M3 (C5)', m3, 3), ('M4 (Grötzsch)', m4, 4)]:
    t0 = time.time()
    chi, _ = chromatic_number(g, max_k=expected_chi+1)
    h = hadwiger_number(g, max_k=expected_chi+2)
    elapsed = time.time()-t0
    print(f'{name:12s}  {len(g):>4}  {chi:>4}  {h:>6}  {str(h>=chi):>8}  {elapsed:.2f}s')

# M5: only check h(G), don't compute chi (too slow)
print(f'{"M5":12s}  {len(m5):>4}  {"5*":>4}  ', end='')
t0 = time.time()
h_m5 = hadwiger_number(m5, max_k=6)
elapsed = time.time()-t0
print(f'{h_m5:>6}  {str(h_m5>=5):>8}  {elapsed:.2f}s')
print()
print('* M5 chi computed theoretically; brute-force too slow for n=23')
print('  Conjecture check uses h(G) >= 5 (theoretical chi=5)')

In [ ]:
# Draw M3 and M4 with their colourings
# M4 (Grötzsch) is famous — 11 vertices, triangle-free, needs 4 colours

chi_m3, col_m3 = chromatic_number(m3, max_k=4)
chi_m4, col_m4 = chromatic_number(m4, max_k=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Mycielski graphs — triangle-free but high chromatic number', fontsize=12)

draw_graph(m3, col_m3, f'M3 = C5\nχ={chi_m3}, h={hadwiger_number(m3,4)}, triangle-free', axes[0])

# Grötzsch graph layout: inner pentagon + outer ring
n4 = len(m4)  # 11 vertices
angles_inner = np.linspace(np.pi/2, np.pi/2+2*np.pi, 5, endpoint=False)
angles_outer = np.linspace(np.pi/2+np.pi/5, np.pi/2+np.pi/5+2*np.pi, 5, endpoint=False)
pos_m4 = np.zeros((n4, 2))
for i in range(5):
    pos_m4[i]   = [0.5*np.cos(angles_inner[i]),  0.5*np.sin(angles_inner[i])]
    pos_m4[i+5] = [np.cos(angles_outer[i]),       np.sin(angles_outer[i])]
pos_m4[10] = [0, 0]  # hub

draw_graph(m4, col_m4,
           f'M4 = Grötzsch graph\nχ={chi_m4}, h={hadwiger_number(m4,5)}, triangle-free',
           axes[1], pos_m4)

plt.tight_layout()
plt.show()

## 4. The computational wall

I've been hitting the wall throughout this series — the 4×4 grid in notebook 2 that ran slowly, the M5 graph above where I had to skip the chromatic number computation. Let me be precise about where the wall is and why.

In [ ]:
# Profile the two bottlenecks separately:
# 1. Chromatic number computation (brute force: k^n colourings)
# 2. Hadwiger number / minor detection (recursive search)

print('Chromatic number computation — brute force cost:')
print(f'{"n":>4}  {"k":>4}  {"Colourings to check":>22}  {"Time (s)":>10}')
print('-' * 46)

for n in range(3, 12):
    g = random_graph(n, 0.5, seed=42)
    t0 = time.time()
    chi, _ = chromatic_number(g, max_k=6)
    elapsed = time.time()-t0
    total_checked = sum(chi**i for i in range(1, n+1))  # approximate
    print(f'{n:>4}  {chi:>4}  {chi**n:>22,}  {elapsed:>10.3f}')
    if elapsed > 10:
        print(f'  (stopping — n={n} already takes {elapsed:.0f}s)')
        break

print()
print('The brute-force search checks up to k^n colourings.')
print('For n=15 and k=4: 4^15 = 1,073,741,824. Not feasible.')

In [ ]:
# Minor detection wall — harder to characterise because it depends
# on graph structure, not just size

print('Minor detection — time varies dramatically by graph structure:')
print(f'{"Graph":25s}  {"n":>4}  {"Seeking":>8}  {"Result":>8}  {"Time (s)":>10}')
print('-' * 62)

test_cases = [
    ('K5',            complete_graph(5),  5),
    ('K6',            complete_graph(6),  6),
    ('C10 (sparse)',  cycle_graph(10),    3),
    ('M4 (Grötzsch)', m4,                 4),
    ('random n=9 p=0.8', random_graph(9, 0.8, 1), 5),
    ('random n=10 p=0.5', random_graph(10, 0.5, 1), 4),
]

for name, g, k in test_cases:
    t0 = time.time()
    result = has_k_minor(g, k, max_depth=14)
    elapsed = time.time()-t0
    print(f'{name:25s}  {len(g):>4}  {f"K{k}":>8}  {str(result):>8}  {elapsed:>10.3f}')

print()
print('Dense graphs: minor found immediately (found early in search tree).')
print('Sparse graphs with high chi: exhaustive search, slow.')
print('This asymmetry matters: the hard cases for Hadwiger are exactly')
print('the sparse high-chromatic graphs — and those are the slow ones.')

In [ ]:
# Visualise the wall: search time as a function of n and density
print('Building timing map (this will take a minute)...')

ns = [5, 6, 7, 8, 9]
ps = [0.3, 0.5, 0.7, 0.9]
K_TARGET = 4
N_SEEDS = 5

timing_grid = np.zeros((len(ns), len(ps)))

for i, n in enumerate(ns):
    for j, p in enumerate(ps):
        times = []
        for seed in range(N_SEEDS):
            g = random_graph(n, p, seed)
            t0 = time.time()
            has_k_minor(g, K_TARGET, max_depth=12)
            times.append(time.time()-t0)
        timing_grid[i,j] = np.mean(times)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(timing_grid, aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(ps)))
ax.set_yticks(range(len(ns)))
ax.set_xticklabels([f'p={p}' for p in ps])
ax.set_yticklabels([f'n={n}' for n in ns])
ax.set_xlabel('Edge density', fontsize=10)
ax.set_ylabel('Number of vertices', fontsize=10)
ax.set_title(f'Mean time to detect K{K_TARGET} minor (seconds)\nGreen=fast, Red=slow', fontsize=11)

for i in range(len(ns)):
    for j in range(len(ps)):
        ax.text(j, i, f'{timing_grid[i,j]:.3f}', ha='center', va='center',
                fontsize=9, color='black')

plt.colorbar(im, ax=ax, label='seconds')
plt.tight_layout()
plt.show()

print()
print('Sparse graphs (low p) take longer regardless of n.')
print('Dense graphs: minor found in the first few contractions.')
print('This is why the conjecture is hard to verify computationally:')
print('the cases that matter most (sparse, high-chi) are the slowest.')

## 5. What the computational evidence tells us

Three things stand out from the experiments:

**The conjecture is robustly true in the cases we can check.** 100 random graphs, 0 violations. The Mycielski graphs — the most adversarial family — satisfy the conjecture. Not once, in any trial, has a graph appeared where χ(G) > h(G).

**The tight cases are interesting.** About a third of random 8-vertex graphs have h(G) = χ(G) exactly. These graphs have less room for error — they're the ones where the conjecture is most constraining. They tend to be sparser than the comfortable cases, which is consistent with the Mycielski observation: it's the sparse graphs where the relationship between colouring and minor structure is tightest.

**The search is harder where it matters most.** Dense graphs reveal their minors quickly — the first few contractions usually suffice. Sparse high-chromatic graphs require exhaustive search. And the hard cases for Hadwiger's Conjecture are exactly those sparse high-chromatic graphs. There's a fundamental mismatch between what the conjecture needs us to verify and what's computationally easy to verify.

This is the computational version of the theoretical wall: even if we wanted to prove the conjecture by checking cases, we'd be slowed down most by precisely the cases we care about most.

**Next:** Notebook 5 — The wall. Synthesis: what kind of hardness is this, and how does it compare to Collatz?